# ClipGenius Cutter - Google Colab (untuk HP)Jalankan di browser HP, gratis, tanpa install apa pun.**Cara pakai:**1. Upload file ini ke Google Colab (colab.research.google.com)2. Jalankan sel **① Persiapan**3. Di sel **②**, tempel link YouTube + JSON klip dari website ClipGenius (tombol Salin JSON)4. Jalankan sel **③** - tunggu sampai selesai5. Jalankan sel **④** - file ZIP berisi MP4 akan terdownloadCatatan: kalau muncul error bot, pakai opsi cookies (ekspor cookies.txt dari browser, upload ke Colab, tambahkan `--cookies cookies.txt` pada perintah yt-dlp).

In [ ]:
# @title ① Persiapan!pip -q install -U yt-dlpimport json, osos.makedirs('/content/hasil_klip', exist_ok=True)print('Siap!')

In [ ]:
# @title ② Tempel data dari websiteYOUTUBE_URL = "https://www.youtube.com/watch?v=xxxxx"  # @param {type:"string"}CLIPS_JSON  = """[{\"start_sec\":245,\"end_sec\":303,\"score\":95,\"type\":\"komedi\",\"title\":\"Klip 1\"}]"""  # @param {type:"string"}VERTICAL = True   # True = 9:16 untuk Shortsclips = json.loads(CLIPS_JSON)if isinstance(clips, dict):    clips = clips.get('clips', [])print(str(len(clips)) + ' klip siap dipotong')

In [ ]:
# @title ③ Unduh & potongimport subprocess, redef hhmmss(s):    s = max(0, int(s))    h, s = divmod(s, 3600)    m, s = divmod(s, 60)    return '%02d:%02d:%02d' % (h, m, s) if h else '%02d:%02d' % (m, s)def safe(n, L=50):    n = re.sub(r'[^\w\s-]', '', str(n))    n = re.sub(r'\s+', ' ', n).strip()    return n[:L] or 'klip'print('Mengunduh video...')r = subprocess.run(['yt-dlp', '-f', 'bestvideo[height<=1080][ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best',                    '-o', '/content/src.%(ext)s', '--no-playlist', '--merge-output-format', 'mp4', YOUTUBE_URL],                   capture_output=True, text=True)if r.returncode != 0:    print(r.stderr[-1200:])    raise SystemExit('Gagal unduh - coba pakai cookies atau ganti video')src = '/content/' + [f for f in os.listdir('/content') if f.startswith('src.')][0]print('Video OK:', src)made = []for i, c in enumerate(clips, 1):    st = max(0, float(c['start_sec']) - 2)    en = float(c['end_sec']) + 2    name = safe('%02d_%s_%s_%s' % (i, c.get('score', 0), c.get('type', 'klip'), c.get('title', ''))) + '.mp4'    out = '/content/hasil_klip/' + name    cmd = ['ffmpeg', '-y', '-hide_banner', '-loglevel', 'error', '-ss', hhmmss(st), '-i', src, '-t', str(int(en - st))]    if VERTICAL:        cmd += ['-vf', 'crop=ih*9/16:ih,scale=1080:1920:flags=lanczos,setsar=1']    cmd += ['-c:v', 'libx264', '-preset', 'veryfast', '-crf', '20', '-c:a', 'aac', '-b:a', '128k', '-movflags', '+faststart', out]    print('  [%d/%d] %s' % (i, len(clips), str(c.get('title', ''))[:40]))    rr = subprocess.run(cmd, capture_output=True, text=True)    if rr.returncode == 0 and os.path.exists(out):        made.append(out)        print('     OK ->', name, '%.1f MB' % (os.path.getsize(out) / 1048576))    else:        print('     GAGAL:', rr.stderr[:200])print('')print('SELESAI: %d/%d klip' % (len(made), len(clips)))

In [ ]:
# @title ④ Download hasil (ZIP)import zipfilez = '/content/hasil_klip.zip'with zipfile.ZipFile(z, 'w') as zf:    for f in made:        zf.write(f, os.path.basename(f))print('ZIP:', z, '%.1f MB' % (os.path.getsize(z) / 1048576))try:    from google.colab import files    files.download(z)except ImportError:    print('Bukan di Colab - file ada di', z)